In [1]:
import os
for _var in ("VECLIB_MAXIMUM_THREADS", "OPENBLAS_NUM_THREADS",
             "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[_var] = "1"

import time
import multiprocessing as mp

import numpy as np

from thotgrad import MLP, Tensor
from thotopt import AdamW, lbfgs

# --- the Dirac 1+1D stationary problem (verbatim reduction, NumPy only) -----
class DiracProblem:
    name = "Dirac (1+1D)"
    def __init__(self, E=2.0, m=1.0, L=6.0, n_col=400):
        assert E > m
        self.E, self.m, self.L = E, m, L
        self.k = float(np.sqrt(E ** 2 - m ** 2))
        self.x = np.linspace(0.0, L, n_col, dtype=float).reshape(-1, 1)
        self.x0 = np.array([0.0]).reshape(-1, 1)
    def exact(self, x):
        x = np.asarray(x).ravel()
        return np.cos(self.k * x), -(self.k / (self.E + self.m)) * np.sin(self.k * x)
    def loss_fn(self, model):
        E, m = self.E, self.m
        xp, x0p = Tensor(self.x), Tensor(self.x0)
        s1 = np.array([[1.0], [0.0]]); s2 = np.array([[0.0], [1.0]])
        def closure():
            out, dout = model.forward_with_derivs(xp, order=1)
            u, v = out @ Tensor(s1), out @ Tensor(s2)
            du, dv = dout @ Tensor(s1), dout @ Tensor(s2)
            r1 = du - Tensor(E + m) * v
            r2 = dv + Tensor(E - m) * u
            phys = (r1 * r1).mean() + (r2 * r2).mean()
            o0 = model(x0p)
            b_u = o0 @ Tensor(s1) - Tensor(1.0)
            b_v = o0 @ Tensor(s2)
            bc = (b_u * b_u).mean() + (b_v * b_v).mean()
            return phys + Tensor(10.0) * bc
        return closure

def make_model(seed=0):
    return MLP(n_in=1, n_out=2, width=64, depth=4, seed=seed)   # 12,738 params


In [2]:
prob = DiracProblem(n_col=400)
model = make_model(0)
loss_fn = prob.loss_fn(model)

from pinn_core import train_pinn
res = train_pinn(model, loss_fn, n_adam=3000, n_lbfgs=500, lr=1e-3,
                 log_every=1000, verbose=True)
t_adam, t_lbfgs = res["t_adam"], res["t_lbfgs"]
n_lb = max(len(res["history"]) - 3001, 1)

u_ex, v_ex = prob.exact(prob.x.ravel())
out = model(Tensor(prob.x)).data
rel_u = float(np.linalg.norm(out[:, 0] - u_ex) / np.linalg.norm(u_ex))
rel_v = float(np.linalg.norm(out[:, 1] - v_ex) / np.linalg.norm(v_ex))
joint = float(np.sqrt(np.linalg.norm(out[:, 0] - u_ex) ** 2 +
                      np.linalg.norm(out[:, 1] - v_ex) ** 2) /
              np.sqrt(np.linalg.norm(u_ex) ** 2 + np.linalg.norm(v_ex) ** 2))
print(f"[AdamW 3000 it] {t_adam:6.1f} s        (case-study anchor: 51.1 s)")
print(f"[L-BFGS {n_lb} it] {t_lbfgs:6.1f} s        (anchor: 10.3 s)")
print(f"final loss            : {res['final_loss']:.4e}  (anchor: 3.74e-04)")
print(f"rel-L2 (u, v, joint)  : {rel_u:.2e} / {rel_v:.2e} / {joint:.2e}"
      "   (anchor: 3.33e-02 joint)")


  [AdamW]     0  loss = 1.0430e+01
  [AdamW]  1000  loss = 3.2239e-02
  [AdamW]  2000  loss = 2.4109e-02
  [AdamW]  2999  loss = 1.4484e-02
  [LBFGS] 500 iterations, final loss = 3.7380e-04
[AdamW 3000 it]   51.6 s        (case-study anchor: 51.1 s)
[L-BFGS 500 it]   10.8 s        (anchor: 10.3 s)
final loss            : 3.7380e-04  (anchor: 3.74e-04)
rel-L2 (u, v, joint)  : 3.26e-02 / 3.57e-02 / 3.33e-02   (anchor: 3.33e-02 joint)


In [3]:
def hidden_chain(model, x, need_deriv=False):
    h = np.asarray(x, dtype=float).reshape(-1, 1)
    dh = np.ones_like(h) if need_deriv else None
    for W, b in model.layers[:-1]:
        augW = np.concatenate([W.data, b.data], axis=0)
        a = np.concatenate([h, np.ones((h.shape[0], 1))], axis=1) @ augW
        if need_deriv:
            da = np.concatenate([dh, np.zeros((dh.shape[0], 1))], axis=1) @ augW
        h = np.tanh(a)
        if need_deriv:
            dh = (1.0 - h ** 2) * da
    return (h, dh) if need_deriv else (h,)

def gn_last_layer(model, prob):
    t0 = time.perf_counter()
    E, m = prob.E, prob.m
    x = prob.x.ravel()
    h4, dh4 = hidden_chain(model, x, need_deriv=True)
    Nn, wid = h4.shape
    hd = np.concatenate([h4, np.ones((Nn, 1))], axis=1)
    hdd = np.concatenate([dh4, np.zeros((Nn, 1))], axis=1)
    A = np.zeros((2 * Nn + 2, 2 * (wid + 1)))
    b_t = np.zeros((2 * Nn + 2,))
    A[:Nn, :wid + 1] = hdd
    A[:Nn, wid + 1:] = -(E + m) * hd
    A[Nn:2 * Nn, wid + 1:] = hdd
    A[Nn:2 * Nn, :wid + 1] = (E - m) * hd
    h0 = hidden_chain(model, prob.x0.ravel())[0]
    hd0 = np.concatenate([h0, np.ones((prob.x0.shape[0], 1))], axis=1)
    A[2 * Nn, :wid + 1] = np.sqrt(10.0) * hd0
    A[2 * Nn + 1, wid + 1:] = np.sqrt(10.0) * hd0
    b_t[2 * Nn] = np.sqrt(10.0)
    th2, *_ = np.linalg.lstsq(A, b_t, rcond=None)
    W5, b5 = model.layers[-1]
    W5.data = np.stack([th2[:wid], th2[wid + 1:2 * wid + 1]], axis=1).copy()
    b5.data = np.array([[th2[wid], th2[2 * wid + 1]]])
    return time.perf_counter() - t0, A.shape

loss_before = float(loss_fn().data)
t_gn, Ash = gn_last_layer(model, prob)
loss_after = float(loss_fn().data)
out = model(Tensor(prob.x)).data
rel_after = float(np.sqrt(np.linalg.norm(out[:, 0] - u_ex) ** 2 +
                          np.linalg.norm(out[:, 1] - v_ex) ** 2) /
                  np.sqrt(np.linalg.norm(u_ex) ** 2 + np.linalg.norm(v_ex) ** 2))
per_it_lbfgs = t_lbfgs / n_lb
print(f"A shape (rows x cols) : {Ash[0]} x {Ash[1]}")
print(f"loss before GN        : {loss_before:.4e}")
print(f"loss after  GN        : {loss_after:.4e}"
      f"   (before/after ratio = {loss_before / max(loss_after, 1e-300):.3g})")
print(f"rel-L2 after GN       : {rel_after:.2e}")
print(f"GN wall-clock         : {t_gn*1000.0:.1f} ms")
print(f"L-BFGS wall per iter  : {per_it_lbfgs*1000.0:.1f} ms  "
      f"-> GN step costs {t_gn / per_it_lbfgs:.2f} L-BFGS iterations")


A shape (rows x cols) : 802 x 130
loss before GN        : 3.7380e-04
loss after  GN        : 2.4600e-21   (before/after ratio = 1.52e+17)
rel-L2 after GN       : 2.76e-12
GN wall-clock         : 12.5 ms
L-BFGS wall per iter  : 21.6 ms  -> GN step costs 0.58 L-BFGS iterations


In [4]:
def loss_and_grad_given(E, m, x0, xp, theta_flat):
    mdl = make_model(0)
    mdl.set_flat(theta_flat)
    x = Tensor(xp.reshape(-1, 1))
    s1 = np.array([[1.0], [0.0]]); s2 = np.array([[0.0], [1.0]])
    def closure():
        out, dout = mdl.forward_with_derivs(x, order=1)
        u, v = out @ Tensor(s1), out @ Tensor(s2)
        du, dv = dout @ Tensor(s1), dout @ Tensor(s2)
        r1 = du - Tensor(E + m) * v
        r2 = dv + Tensor(E - m) * u
        o0 = mdl(Tensor(x0))
        b_u = o0 @ Tensor(s1) - Tensor(1.0)
        b_v = o0 @ Tensor(s2)
        return ((r1 * r1).mean() + (r2 * r2).mean()
                + Tensor(10.0) * ((b_u * b_u).mean() + (b_v * b_v).mean()))
    L = closure()
    L.backward()
    return float(L.data), mdl.grad_flat()

def _worker(pack):
    E, m, x0, th, xs = pack
    return loss_and_grad_given(E, m, x0, xs, th)

import sys
sys.modules["__main__"]._worker = _worker   # picklable top-level name for Pool

prob_d = DiracProblem(n_col=1600)
xd = prob_d.x.ravel()
th0 = make_model(0).get_flat()
l_ref, g_ref = loss_and_grad_given(prob_d.E, prob_d.m, prob_d.x0, xd, th0)
rows = []
for w in (1, 2, 4):
    n_sl = 1600 // w
    packs = [(prob_d.E, prob_d.m, prob_d.x0, th0,
              xd[i * n_sl:(i + 1) * n_sl]) for i in range(w)]
    ctx = mp.get_context("fork")
    with ctx.Pool(w) as pool:
        t0 = time.perf_counter()
        res = pool.map(_worker, packs)
        s = time.perf_counter() - t0
    l_mean = sum(li for li, _ in res) / w
    g_mean = np.sum([gi for _, gi in res], axis=0) / w
    rel = float(np.max(np.abs(g_mean - g_ref))) / float(np.max(np.abs(g_ref)))
    rows.append((w, s, rel))
print(f"full-batch gradient norm: {float(np.max(np.abs(g_ref))):.3e}")
print(f"{'workers':>7} {'wall/epoch':>12} {'max rel grad dev':>18}")
for w, s, rel in rows:
    print(f"{w:>7} {s*1000.0:>10.0f}ms {rel:>18.2e}   (round-off: distribution exact)")


full-batch gradient norm: 1.931e+01
workers   wall/epoch   max rel grad dev
      1         82ms           0.00e+00   (round-off: distribution exact)
      2         55ms           4.14e-16   (round-off: distribution exact)
      4         34ms           2.99e-16   (round-off: distribution exact)


In [5]:
rng = np.random.default_rng(20261018)
data = rng.normal(2.0, 0.8, 200)
vbar, m2_ = float(data.mean()), float((data ** 2).mean())

def risk(mu, sig):
    return float(np.log(sig) + (m2_ - 2 * mu * vbar + mu * mu) / (2 * sig ** 2))

def grads(mu, sig):
    SS = m2_ - 2 * mu * vbar + mu * mu
    return (mu - vbar) / sig ** 2, 1.0 / sig - SS / sig ** 3

# -- certificate 1: direction invariance (pushed through the chart) ---------
mu0, sig0 = 1.4, 0.5
dmu, dsig = grads(mu0, sig0)
dir_sigma = np.array([sig0 ** 2 * dmu, (sig0 ** 2 / 2.0) * dsig])   # sigma chart
dsig_from_s = sig0 * (0.5 * (dsig * sig0))                          # s chart via J: d sigma = sigma d s
dir_s_pushed = np.array([sig0 ** 2 * dmu, dsig_from_s])
print("natural-gradient direction, sigma chart vs log-chart pushed through J:")
print("  max component deviation =", f"{float(np.max(np.abs(dir_sigma - dir_s_pushed))):.1e}",
      " (float64 round-off: same tangent vector)")

# -- certificate 2: discrete 40-step paths in both charts -------------------
def run(method, chart, eta, n_it=40, mu=-3.5, sig=0.6):
    for _ in range(n_it):
        dmu, dsig = grads(mu, sig)
        if method == "gd" and chart == "log":          # dL/ds = sigma dL/dsigma
            mu -= eta * dmu
            sig = float(np.exp(np.log(sig) - eta * dsig * sig))
        elif method == "gd":
            mu -= eta * dmu
            sig -= eta * dsig
        else:                                          # natural gradient
            if chart == "log":                         # native (mu, log sigma) chart
                s = np.log(sig)
                mu -= eta * sig ** 2 * dmu             # F'^{-1}_mumu = sigma^2
                s -= eta * 0.5 * (dsig * sig)          # F'^{-1}_ss   = 1/2
                sig = float(np.exp(s))
            else:                                      # native (mu, sigma) chart
                mu -= eta * sig ** 2 * dmu             # F^{-1}_mumu  = sigma^2
                sig -= eta * (sig ** 2 / 2.0) * dsig   # F^{-1}_sigsig = sigma^2/2
    return risk(mu, sig), mu, sig

eta = 0.06
res = {f"{m}_{c}": run(m, c, eta) for m in ("gd", "ng")
       for c in ("log", "lin")}
dev_ng = abs(res["ng_log"][1] - res["ng_lin"][1]) + abs(res["ng_log"][2] - res["ng_lin"][2])
dev_gd = abs(res["gd_log"][1] - res["gd_lin"][1]) + abs(res["gd_log"][2] - res["gd_lin"][2])
print(f"{'method/chart':>14} {'final risk':>12} {'final (mu, sigma)':>24}")
for name, (r, mu_f, sig_f) in res.items():
    print(f"{name:>14} {r:>12.5f}   ({mu_f:+8.4f}, {sig_f:7.4f})")
print(f"discrete NG path drift over 40 steps   : {dev_ng:.3f}   (O(eta) per step, as documented)")
print(f"discrete GD path drift over 40 steps   : {dev_gd:.3f}   (no geometry protection: measured drift)")


natural-gradient direction, sigma chart vs log-chart pushed through J:
  max component deviation = 0.0e+00  (float64 round-off: same tangent vector)
  method/chart   final risk        final (mu, sigma)
        gd_log      2.52878   ( -2.5556, 11.5437)
        gd_lin      2.33510   ( -2.4467,  9.0980)
        ng_log      1.16802   ( +1.5924,  3.0587)
        ng_lin      0.75944   ( +1.5924,  1.8692)
discrete NG path drift over 40 steps   : 1.190   (O(eta) per step, as documented)
discrete GD path drift over 40 steps   : 2.555   (no geometry protection: measured drift)


In [6]:
print("Case-study walkthrough complete: protocol, geometry, distribution, "
      "and invariance all verified on one CPU-only host.")


Case-study walkthrough complete: protocol, geometry, distribution, and invariance all verified on one CPU-only host.
